# Visual Question Answering -- BLIP and CLIP

This notebook fine-tunes `Salesforce/blip-vqa-base` on the `Visual Question Answering- Computer Vision & NLP` dataset
as a multi-class classification task (582 answer classes).

### Approach and Design Decisions

The task is framed as a **582-class multi-class classification** problem. The following design steps shape the pipeline:

- **Multi-label answer normalisation:** `_resolve_answer()` splits comma-separated answer strings (e.g. `"picture, wall_decoration"`) and picks one valid sub-token from the vocabulary. During training a sub-token is chosen randomly (label augmentation); during evaluation the first valid sub-token is always used for deterministic scoring.
- **3-layer MLP head:** The generative decoder of `blip-vqa-base` is discarded. The `[CLS]` token of the multimodal encoder is fed through `Linear(768,512) → LayerNorm → GELU → Dropout(0.2) → Linear(512, C)`, providing sufficient classification capacity with built-in regularisation.
- **Progressive vision unfreezing:** The ViT encoder is frozen for the first 5 epochs to stabilise the randomly-initialised head, then unfrozen and added to the optimiser at 0.1× base learning rate.
- **Cosine LR schedule with warm-up:** AdamW (`lr=4e-5`, `weight_decay=0.01`) with cosine decay and 10% linear warmup, stepped once per optimiser update (every 8 gradient accumulation steps).
- **Class-weighted loss:** Inverse-frequency class weights capped at 5× are passed to `CrossEntropyLoss` with label smoothing of 0.1, handling 806 singleton answer classes in the training split.
- **AMP + gradient accumulation (8 steps):** Mixed precision training with an effective batch size of 64 without exceeding T4 VRAM limits.
- **Training-time data augmentation:** `RandomHorizontalFlip`, `ColorJitter`, and `RandomResizedCrop` are applied to each PIL image before the BLIP processor, adding variety from the 1,449 unique images in the dataset.
- **Best-only checkpointing:** A single `BLIP_best.pt` file is kept, replaced only when validation accuracy improves, staying within Kaggle's 20 GB disk limit.
- **Per-epoch CSV logging:** Each epoch immediately appends its metrics (loss, accuracy, F1, BLEU) to `VQAExperiments.csv` with a unique `{uuid}_ep{epoch}` row identifier.

### Strategy Pattern
`ModelStrategy` (ABC) → `BLIPStrategy` or `CLIPStrategy`. Set `CONFIG["model_choice"]`.

### Pipeline
1. Install dependencies
2. Define all classes (Strategy Pattern with both BLIP and CLIP)
3. Explore dataset
4. Load data, compute class weights
5. Train: AMP + gradient accumulation + cosine LR
6. Evaluate per epoch (Accuracy, F1, BLEU) — one CSV row per epoch
7. Save predictions (GeneratedAnswers) and training-curve plots

In [1]:
import subprocess, sys

packages = [
    "transformers>=4.35.0",
    "accelerate>=0.24.0",
    "scikit-learn>=1.3.0",
    "nltk>=3.8.0",
]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Packages ready.")

Packages ready.


In [2]:
import nltk

# punkt_tab is required on NLTK >= 3.8 (Kaggle ships this version)
for resource in ("punkt", "punkt_tab"):
    try:
        nltk.download(resource, quiet=True)
    except Exception:
        pass

print("NLTK data ready.")

NLTK data ready.


In [3]:
import csv
import json
import os
import random
import uuid
import warnings
from abc import ABC, abstractmethod
from collections import Counter
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BlipForQuestionAnswering,
    BlipProcessor,
    CLIPModel,
    CLIPProcessor,
    get_cosine_schedule_with_warmup,
)

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU      : {props.name}")
    print(f"VRAM     : {props.total_memory / 1e9:.1f} GB")

PyTorch  : 2.9.0+cu126
CUDA     : True
GPU      : Tesla T4
VRAM     : 15.6 GB


In [4]:
# -----------------------------------------------------------------------
# CONFIGURATION 
# -----------------------------------------------------------------------

DATASET_DIR     = "/kaggle/input/datasets/bhavikardeshna/visual-question-answering-computer-vision-nlp/dataset"
OUTPUT_DIR      = "/kaggle/working/outputs"
EXPERIMENTS_CSV = "/kaggle/working/VQAExperiments.csv"

# Model selection: "blip" or "clip"
MODEL_CHOICE = "blip"

CONFIG = {
    # ---- model ----
    "model_choice"               : MODEL_CHOICE,
    "model_name"                 : MODEL_CHOICE.upper(),
    "pretrained_id"              : (
        "Salesforce/blip-vqa-base"
        if MODEL_CHOICE == "blip"
        else "openai/clip-vit-base-patch32"
    ),
    # ---- training ----
    "learning_rate"              : 4e-5,   # was 5e-5; lower is safer with small dataset
    "batch_size"                 : 8,
    "epochs"                     : 15,
    "gradient_accumulation_steps": 8,      # was 4; effective batch = 64 for better grad estimates
    "max_length"                 : 32,
    "weight_decay"               : 0.01,
    "label_smoothing"            : 0.1,
    "warmup_ratio"               : 0.1,
    "max_grad_norm"              : 1.0,
    # ---- regularisation ----
    "freeze_vision_epochs"       : 5,      # was 2; give head more time with only 9,974 samples
    "classifier_dropout"         : 0.2,   # was 0.3; head is already regularised by weight_decay
    "class_weight_cap"           : 5.0,    # was 10.0; 10x caused large loss spikes on rare classes
    # ---- hardware ----
    "mixed_precision"            : True,
    "seed"                       : 42,
    # ---- paths ----
    "checkpoint_dir"             : os.path.join(OUTPUT_DIR, "checkpoints"),
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)
print("Configuration:")
print(json.dumps({k: v for k, v in CONFIG.items() if "dir" not in k}, indent=2))

Configuration:
{
  "model_choice": "blip",
  "model_name": "BLIP",
  "pretrained_id": "Salesforce/blip-vqa-base",
  "learning_rate": 4e-05,
  "batch_size": 8,
  "epochs": 15,
  "gradient_accumulation_steps": 8,
  "max_length": 32,
  "weight_decay": 0.01,
  "label_smoothing": 0.1,
  "warmup_ratio": 0.1,
  "max_grad_norm": 1.0,
  "freeze_vision_epochs": 5,
  "classifier_dropout": 0.2,
  "class_weight_cap": 5.0,
  "mixed_precision": true,
  "seed": 42
}


In [5]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def compute_class_weights(
    dataframe: pd.DataFrame,
    answer2label: Dict[str, int],
    num_classes: int,
    cap: float = 3.0,
) -> torch.Tensor:
    """
    Inverse-frequency class weights to handle the 582-class imbalance.
    Cap defaults to 5.0 (was 10.0).  The aggressive 10x cap was causing
    large loss spikes whenever a batch was dominated by singleton-class
    samples, destabilising training.
    """
    labels = []
    for ans in dataframe["answer"].astype(str):
        lbl = answer2label.get(ans, -1)
        if 0 <= lbl < num_classes:
            labels.append(lbl)
    counts = Counter(labels)
    total = len(labels)
    weights = torch.ones(num_classes)
    for cls, cnt in counts.items():
        weights[cls] = total / (num_classes * cnt)
    return weights.clamp(max=cap)


def plot_training_curves(
    train_loss: List[float],
    val_loss: List[float],
    accuracy: List[float],
    output_dir: str,
    model_name: str = "model",
) -> str:
    epochs = list(range(1, len(train_loss) + 1))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f"Training Curves -- {model_name}", fontsize=14, fontweight="bold")

    axes[0].plot(epochs, train_loss, marker="o", color="tab:blue", label="Train")
    axes[0].set(title="Training Loss vs Epoch", xlabel="Epoch", ylabel="Loss")
    axes[0].legend()
    axes[0].grid(True, linestyle="--", alpha=0.5)

    axes[1].plot(epochs, val_loss, marker="s", color="tab:orange", label="Val")
    axes[1].set(title="Validation Loss vs Epoch", xlabel="Epoch", ylabel="Loss")
    axes[1].legend()
    axes[1].grid(True, linestyle="--", alpha=0.5)

    axes[2].plot(epochs, accuracy, marker="^", color="tab:green", label="Accuracy")
    axes[2].set(title="Accuracy vs Epoch", xlabel="Epoch", ylabel="Accuracy")
    axes[2].set_ylim(0, 1)
    axes[2].legend()
    axes[2].grid(True, linestyle="--", alpha=0.5)

    plt.tight_layout()
    path = os.path.join(output_dir, f"{model_name}_training_curves.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Plot saved to '{path}'")
    return path


set_seed(CONFIG["seed"])
print("Utilities defined and seed set.")

Utilities defined and seed set.


In [6]:
import torchvision.transforms as T

# Applied to PIL images during TRAINING only -- not eval.
# Only 1,449 unique images in this dataset so augmentation has high value.
_TRAIN_AUGMENT = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    T.RandomResizedCrop(size=224, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
])


class VQADataset(Dataset):
    """
    PyTorch Dataset for VQA multi-class classification.

    Edge-case handling
    ------------------
    Missing images   : replaced with a blank (black) RGB image.

    Comma-separated answers (e.g. "picture, wall_decoration"):
        During TRAINING  -> randomly picks one valid token (augmentation).
        During EVAL/TEST -> always picks the first valid token (deterministic).

    Image augmentation (training only):
        RandomHorizontalFlip + ColorJitter + RandomResizedCrop.
        Since there are only 1,449 unique images shared across ~7 questions
        each, augmentation provides significant extra variety.
    """

    def __init__(
        self,
        dataframe: pd.DataFrame,
        image_dir: str,
        answer2label: Dict[str, int],
        processor: Any,
        max_length: int = 32,
        is_training: bool = False,
    ) -> None:
        self.dataframe    = dataframe.reset_index(drop=True)
        self.image_dir    = image_dir
        self.answer2label = answer2label
        self.processor    = processor
        self.max_length   = max_length
        self.is_training  = is_training

        unresolvable = [
            str(a) for a in dataframe["answer"]
            if self._resolve_answer(str(a), training=False) == 0
            and str(a) not in answer2label
            and not any(
                part.strip() in answer2label
                for part in str(a).split(",")
            )
        ]
        if unresolvable:
            print(
                f"[VQADataset] NOTE: {len(unresolvable)} samples fall back "
                f"to label 0. Examples: {list(set(unresolvable))[:3]}"
            )

    def _resolve_answer(self, answer: str, training: bool = False) -> int:
        """
        Map an answer string to an integer label.

        Priority:
        1. Exact match (fast path, ~90% of data).
        2. Comma-split tokens:
             training=True  -> shuffle tokens, pick first valid (augmentation)
             training=False -> scan left-to-right, pick first valid (deterministic)
        3. Fallback: label 0.
        """
        answer = answer.strip()
        if answer in self.answer2label:
            return self.answer2label[answer]
        parts = [p.strip() for p in answer.split(",") if p.strip()]
        if training:
            random.shuffle(parts)
        for part in parts:
            if part in self.answer2label:
                return self.answer2label[part]
        return 0

    def __len__(self) -> int:
        return len(self.dataframe)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.dataframe.iloc[idx]
        image_path = os.path.join(self.image_dir, f"{row['image_id']}.png")

        image = (
            Image.open(image_path).convert("RGB")
            if os.path.exists(image_path)
            else Image.new("RGB", (224, 224), color=(0, 0, 0))
        )

        # Apply augmentation to PIL image BEFORE the processor
        if self.is_training:
            image = _TRAIN_AUGMENT(image)

        question = str(row["question"])
        answer   = str(row["answer"])
        label    = self._resolve_answer(answer, training=self.is_training)

        encoding = self.processor(
            images=image,
            text=question,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
        )

        return {
            "pixel_values"  : encoding["pixel_values"].squeeze(0),
            "input_ids"     : encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "answer_label"  : torch.tensor(label, dtype=torch.long),
            "image_id"      : str(row["image_id"]),
            "question"      : question,
            "answer"        : answer,
        }


print("VQADataset defined.")

VQADataset defined.


In [7]:
class DatasetProcessor:
    """
    Loads answer_space.txt, data_train.csv, data_eval.csv.
    Exposes VQADataset/DataLoader objects and raw DataFrames
    (required externally for compute_class_weights).
    """

    def __init__(
        self,
        data_dir: str,
        processor: Any,
        batch_size: int = 8,
        num_workers: int = 2,
        max_length: int = 32,
    ) -> None:
        self.data_dir    = data_dir
        self.processor   = processor
        self.batch_size  = batch_size
        self.num_workers = num_workers
        self.max_length  = max_length

        self.image_dir         = os.path.join(data_dir, "images")
        self.answer_space_path = os.path.join(data_dir, "answer_space.txt")
        self.train_csv         = os.path.join(data_dir, "data_train.csv")
        self.eval_csv          = os.path.join(data_dir, "data_eval.csv")

        self.answer2label: Dict[str, int] = {}
        self.label2answer: Dict[int, str] = {}
        self._load_answer_space()

        self.train_df = self._load_df(self.train_csv)
        self.eval_df  = self._load_df(self.eval_csv)

    def _load_answer_space(self) -> None:
        with open(self.answer_space_path, "r", encoding="utf-8") as f:
            answers = [l.strip() for l in f if l.strip()]
        self.answer2label = {a: i for i, a in enumerate(answers)}
        self.label2answer = {i: a for i, a in enumerate(answers)}

    def get_num_classes(self) -> int:
        return len(self.answer2label)

    def _load_df(self, path: str) -> pd.DataFrame:
        df = pd.read_csv(path)
        required = {"question", "answer", "image_id"}
        missing  = required - set(df.columns)
        if missing:
            raise ValueError(f"'{path}' is missing columns: {missing}")
        df["answer"] = df["answer"].astype(str)
        return df

    def get_train_loader(self) -> DataLoader:
        ds = VQADataset(
            self.train_df, self.image_dir,
            self.answer2label, self.processor, self.max_length,
            is_training=True,   # augmentation + random label sampling ON
        )
        return DataLoader(
            ds, batch_size=self.batch_size, shuffle=True,
            num_workers=self.num_workers,
            pin_memory=torch.cuda.is_available(),
        )

    def get_eval_loader(self) -> DataLoader:
        ds = VQADataset(
            self.eval_df, self.image_dir,
            self.answer2label, self.processor, self.max_length,
            is_training=False,  # deterministic, no augmentation
        )
        return DataLoader(
            ds, batch_size=self.batch_size, shuffle=False,
            num_workers=self.num_workers,
            pin_memory=torch.cuda.is_available(),
        )


print("DatasetProcessor defined.")

DatasetProcessor defined.


In [8]:
# -----------------------------------------------------------------------
# BLIP -- Multi-layer classification head on the joint VL encoder output
# -----------------------------------------------------------------------

class BLIPClassificationModel(nn.Module):
    """
    Salesforce/blip-vqa-base adapted for closed-set classification.

    Architecture: Linear(768,512) -> LayerNorm -> GELU -> Dropout -> Linear(512,582)
    Dropout is read from CONFIG["classifier_dropout"] (default 0.2).
    """

    def __init__(
        self,
        blip_model: BlipForQuestionAnswering,
        num_classes: int,
        freeze_vision: bool = True,
        dropout: float = 0.2,
    ) -> None:
        super().__init__()
        self.blip = blip_model

        if freeze_vision:
            self._set_vision_grad(False)

        hidden_size = blip_model.config.text_config.hidden_size  # 768

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )

    def _set_vision_grad(self, requires_grad: bool) -> None:
        for p in self.blip.vision_model.parameters():
            p.requires_grad = requires_grad

    def unfreeze_vision(self) -> None:
        self._set_vision_grad(True)
        print("[BLIPModel] Vision encoder unfrozen.")

    def forward(
        self,
        pixel_values: torch.Tensor,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        vision_out   = self.blip.vision_model(pixel_values=pixel_values)
        image_embeds = vision_out.last_hidden_state

        image_attn = torch.ones(
            image_embeds.size()[:2], dtype=torch.long,
            device=image_embeds.device
        )

        text_out = self.blip.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            encoder_hidden_states=image_embeds,
            encoder_attention_mask=image_attn,
            return_dict=True,
        )

        cls_emb = text_out.last_hidden_state[:, 0, :]
        return self.classifier(cls_emb)


# -----------------------------------------------------------------------
# CLIP -- Frozen backbone + trainable MLP fusion head
# -----------------------------------------------------------------------

class CLIPFusionModel(nn.Module):
    """
    openai/clip-vit-base-patch32 adapted for closed-set VQA.
    CLIP backbone frozen; only fusion MLP is optimised.
    """

    def __init__(self, clip_model: CLIPModel, num_classes: int) -> None:
        super().__init__()
        self.clip = clip_model

        for p in self.clip.parameters():
            p.requires_grad = False

        embed_dim = clip_model.config.projection_dim  # 512

        self.fusion_head = nn.Sequential(
            nn.Linear(embed_dim * 2, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes),
        )

    def forward(
        self,
        pixel_values: torch.Tensor,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        img_emb = self.clip.get_image_features(pixel_values=pixel_values)
        txt_emb = self.clip.get_text_features(
            input_ids=input_ids, attention_mask=attention_mask
        )
        fused = torch.cat([img_emb, txt_emb], dim=-1)
        return self.fusion_head(fused)


print("Model classes (BLIPClassificationModel, CLIPFusionModel) defined.")

Model classes (BLIPClassificationModel, CLIPFusionModel) defined.


In [9]:
# -----------------------------------------------------------------------
# Strategy Pattern
# -----------------------------------------------------------------------

class ModelStrategy(ABC):
    @abstractmethod
    def load_model(self, num_classes: int, device: torch.device) -> None: ...
    @abstractmethod
    def forward(self, batch: Dict[str, Any]) -> torch.Tensor: ...
    @abstractmethod
    def predict(self, batch: Dict[str, Any]) -> torch.Tensor: ...
    @abstractmethod
    def get_model(self) -> nn.Module: ...
    @abstractmethod
    def get_name(self) -> str: ...


class BLIPStrategy(ModelStrategy):
    PRETRAINED_ID = "Salesforce/blip-vqa-base"

    def __init__(self, freeze_vision: bool = True) -> None:
        self._model: Optional[BLIPClassificationModel] = None
        self._device: Optional[torch.device] = None
        self.freeze_vision = freeze_vision

    def load_model(self, num_classes: int, device: torch.device) -> None:
        self._device = device
        print(f"[BLIPStrategy] Loading '{self.PRETRAINED_ID}' ...")
        blip_base   = BlipForQuestionAnswering.from_pretrained(self.PRETRAINED_ID)
        dropout     = CONFIG.get("classifier_dropout", 0.2)
        self._model = BLIPClassificationModel(
            blip_base, num_classes,
            freeze_vision=self.freeze_vision,
            dropout=dropout,
        ).to(device)
        trainable = sum(p.numel() for p in self._model.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in self._model.parameters())
        print(f"[BLIPStrategy] {trainable:,} trainable / {total:,} total params")

    def forward(self, batch: Dict[str, Any]) -> torch.Tensor:
        return self._model(
            pixel_values   = batch["pixel_values"].to(self._device),
            input_ids      = batch["input_ids"].to(self._device),
            attention_mask = batch["attention_mask"].to(self._device),
        )

    def predict(self, batch: Dict[str, Any]) -> torch.Tensor:
        with torch.no_grad():
            return self.forward(batch).argmax(dim=-1)

    def get_model(self) -> nn.Module:
        return self._model

    def get_name(self) -> str:
        return "BLIP"

    def unfreeze_vision(self) -> None:
        if self._model is not None:
            self._model.unfreeze_vision()


class CLIPStrategy(ModelStrategy):
    PRETRAINED_ID = "openai/clip-vit-base-patch32"

    def __init__(self) -> None:
        self._model: Optional[CLIPFusionModel] = None
        self._device: Optional[torch.device] = None

    def load_model(self, num_classes: int, device: torch.device) -> None:
        self._device = device
        print(f"[CLIPStrategy] Loading '{self.PRETRAINED_ID}' ...")
        clip_base   = CLIPModel.from_pretrained(self.PRETRAINED_ID)
        self._model = CLIPFusionModel(clip_base, num_classes).to(device)
        trainable = sum(p.numel() for p in self._model.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in self._model.parameters())
        print(f"[CLIPStrategy] {trainable:,} trainable / {total:,} total params")

    def forward(self, batch: Dict[str, Any]) -> torch.Tensor:
        return self._model(
            pixel_values   = batch["pixel_values"].to(self._device),
            input_ids      = batch["input_ids"].to(self._device),
            attention_mask = batch["attention_mask"].to(self._device),
        )

    def predict(self, batch: Dict[str, Any]) -> torch.Tensor:
        with torch.no_grad():
            return self.forward(batch).argmax(dim=-1)

    def get_model(self) -> nn.Module:
        return self._model

    def get_name(self) -> str:
        return "CLIP"


print("Strategy Pattern (ModelStrategy, BLIPStrategy, CLIPStrategy) defined.")

Strategy Pattern (ModelStrategy, BLIPStrategy, CLIPStrategy) defined.


In [10]:
class Evaluator:
    """Computes Accuracy, weighted F1, and corpus BLEU."""

    def compute(
        self,
        predictions: List[int],
        references: List[int],
        label2answer: Optional[Dict[int, str]] = None,
    ) -> Dict[str, float]:
        if not predictions:
            return {"accuracy": 0.0, "f1": 0.0, "bleu": 0.0}

        return {
            "accuracy": round(float(accuracy_score(references, predictions)), 4),
            "f1"      : round(float(f1_score(
                references, predictions,
                average="weighted", zero_division=0,
            )), 4),
            "bleu"    : round(self._bleu(predictions, references, label2answer), 4),
        }

    def _bleu(
        self,
        preds: List[int],
        refs: List[int],
        label2answer: Optional[Dict[int, str]],
    ) -> float:
        smoother = SmoothingFunction().method1
        scores = []
        for p, r in zip(preds, refs):
            if label2answer:
                p_tok = label2answer.get(p, str(p)).replace("_", " ").split()
                r_tok = label2answer.get(r, str(r)).replace("_", " ").split()
            else:
                p_tok, r_tok = [str(p)], [str(r)]
            scores.append(
                sentence_bleu([r_tok], p_tok, smoothing_function=smoother)
            )
        return sum(scores) / max(len(scores), 1)


print("Evaluator defined.")

Evaluator defined.


In [11]:
CSV_COLUMNS = [
    "id", "model_name", "hyperparameters",
    "train_loss", "val_loss", "metrics", "timestamp",
]


class ExperimentLogger:
    """
    Per-epoch CSV experiment logger.
    """

    def __init__(self, csv_path: str = EXPERIMENTS_CSV) -> None:
        self.csv_path = csv_path
        self._active: Dict[str, Dict] = {}
        if not os.path.exists(csv_path):
            with open(csv_path, "w", newline="", encoding="utf-8") as f:
                csv.DictWriter(f, fieldnames=CSV_COLUMNS).writeheader()

    def start_experiment(self, model_name: str, hyperparameters: Dict) -> str:
        exp_id = str(uuid.uuid4())
        self._active[exp_id] = {
            "model_name"     : model_name,
            "hyperparameters": json.dumps(hyperparameters),
        }
        print(f"[Logger] Experiment started: {exp_id}")
        return exp_id

    def log_epoch(
        self,
        experiment_id: str,
        epoch: int,
        train_loss: float,
        val_loss: float,
        metrics: Dict[str, float],
    ) -> None:
        """Append one row per epoch immediately -- no data lost on crash."""
        base = self._active[experiment_id]
        row = {
            "id"             : f"{experiment_id}_ep{epoch}",
            "model_name"     : base["model_name"],
            "hyperparameters": base["hyperparameters"],
            "train_loss"     : round(float(train_loss), 6),
            "val_loss"       : round(float(val_loss), 6),
            "metrics"        : json.dumps({**metrics, "epoch": epoch}),
            "timestamp"      : datetime.now(tz=timezone.utc).isoformat(),
        }
        with open(self.csv_path, "a", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=CSV_COLUMNS).writerow(row)

        print(
            f"[Logger] Epoch {epoch:>2} | "
            f"train={train_loss:.4f}  val={val_loss:.4f}  "
            f"acc={metrics.get('accuracy', 0):.4f}  "
            f"f1={metrics.get('f1', 0):.4f}  "
            f"bleu={metrics.get('bleu', 0):.4f}"
        )

    def finish_experiment(self, experiment_id: str) -> None:
        self._active.pop(experiment_id, None)
        print(f"[Logger] Complete log -> '{self.csv_path}'")


print("ExperimentLogger defined.")

ExperimentLogger defined.


In [ ]:
class VQAManager:
    """
    Orchestrates training, validation, checkpoint saving, and inference.

    Design decisions
    ----------------
    Cosine LR schedule with 10% linear warmup
        AdamW at lr=4e-5 with cosine decay stepped per optimiser update
        for a smooth learning rate curve across all gradient accumulation steps.
    Label smoothing (0.1) on CrossEntropyLoss
        Prevents overconfident logits; generalises better across 582 classes.
    Inverse-frequency class weights (capped at 5x)
        Addresses the 806 singleton classes in the training split by
        up-weighting rare classes without causing loss spikes on extremes.
    Progressive vision unfreezing
        ViT is frozen for the first 5 epochs so the randomly-initialised head
        reaches a stable optimum before backbone weights are updated. Unfrozen
        vision params are added to the optimiser at 0.1x base LR.
    Per-step scheduler updates
        Scheduler steps after every optimiser update (not once per epoch)
        for a smooth cosine curve.
    Gradient clipping at max_grad_norm
        Prevents exploding gradients when the vision encoder is unfrozen.
    Best-only checkpointing
        Only the single best checkpoint is kept on disk at any time.
        The previous best is deleted before writing the new one, keeping
        peak usage at ~400 MB regardless of epoch count.
    """

    def __init__(
        self,
        strategy: ModelStrategy,
        logger: ExperimentLogger,
        config: Dict[str, Any],
        class_weights: Optional[torch.Tensor] = None,
    ) -> None:
        self.strategy = strategy
        self.logger   = logger
        self.config   = config

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"[Manager] Device: {self.device}")

        self.model    = strategy.get_model()
        trainable     = [p for p in self.model.parameters() if p.requires_grad]
        print(f"[Manager] Trainable params: {sum(p.numel() for p in trainable):,}")

        self.optimizer = AdamW(
            trainable,
            lr=config["learning_rate"],
            weight_decay=config.get("weight_decay", 0.01),
        )

        cw = class_weights.to(self.device) if class_weights is not None else None
        self.criterion = nn.CrossEntropyLoss(
            weight=cw,
            label_smoothing=config.get("label_smoothing", 0.1),
        )

        use_amp       = config["mixed_precision"] and self.device.type == "cuda"
        self.scaler   = GradScaler(enabled=use_amp)
        self._use_amp = use_amp
        self._scheduler = None

        self.train_loss_history: List[float] = []
        self.val_loss_history:   List[float] = []
        self.accuracy_history:   List[float] = []

        # Best-only checkpoint tracking
        self._best_val_acc   = 0.0
        self._best_ckpt_path: Optional[str] = None

    # -----------------------------------------------------------------------
    # Training
    # -----------------------------------------------------------------------

    def train(
        self,
        train_loader: DataLoader,
        val_loader:   DataLoader,
        label2answer: Dict[int, str],
    ) -> Dict[str, List[float]]:
        epochs        = self.config["epochs"]
        accum         = self.config["gradient_accumulation_steps"]
        freeze_epochs = self.config.get("freeze_vision_epochs", 0)

        # Build cosine schedule over total optimizer steps
        steps_per_epoch = int(np.ceil(len(train_loader) / accum))
        total_opt_steps = steps_per_epoch * epochs
        warmup_steps    = int(self.config.get("warmup_ratio", 0.1) * total_opt_steps)
        self._scheduler = get_cosine_schedule_with_warmup(
            self.optimizer, warmup_steps, total_opt_steps
        )
        print(
            f"[Manager] LR scheduler: {total_opt_steps} total steps, "
            f"{warmup_steps} warmup steps"
        )

        exp_id = self.logger.start_experiment(
            self.strategy.get_name(), self.config
        )

        for epoch in range(1, epochs + 1):
            # Unfreeze vision encoder after freeze_vision_epochs
            if freeze_epochs > 0 and epoch == freeze_epochs + 1:
                if hasattr(self.strategy, "unfreeze_vision"):
                    self.strategy.unfreeze_vision()
                    # Add newly-unfrozen vision params at 0.1x base LR
                    existing_ids = {
                        id(p)
                        for group in self.optimizer.param_groups
                        for p in group["params"]
                    }
                    new_params = [
                        p for p in self.model.parameters()
                        if p.requires_grad and id(p) not in existing_ids
                    ]
                    if new_params:
                        self.optimizer.add_param_group({
                            "params": new_params,
                            "lr"    : self.config["learning_rate"] * 0.1,
                        })
                        print(
                            f"[Manager] Added {len(new_params)} vision params "
                            f"at lr={self.config['learning_rate'] * 0.1:.1e}"
                        )

            print(f"\n{'=' * 60}\nEpoch {epoch}/{epochs}\n{'=' * 60}")
            train_loss = self._train_epoch(train_loader, epoch)
            val_loss, metrics = self._val_epoch(val_loader, label2answer)

            self.train_loss_history.append(train_loss)
            self.val_loss_history.append(val_loss)
            self.accuracy_history.append(metrics["accuracy"])

            self._save_ckpt(epoch, metrics["accuracy"])
            self.logger.log_epoch(exp_id, epoch, train_loss, val_loss, metrics)

        self.logger.finish_experiment(exp_id)
        return {
            "train_loss": self.train_loss_history,
            "val_loss"  : self.val_loss_history,
            "accuracy"  : self.accuracy_history,
        }

    def _train_epoch(self, loader: DataLoader, epoch: int) -> float:
        self.model.train()
        total, steps = 0.0, 0
        accum    = self.config["gradient_accumulation_steps"]
        max_norm = self.config.get("max_grad_norm", 1.0)
        self.optimizer.zero_grad()

        for step, batch in enumerate(loader):
            labels = batch["answer_label"].to(self.device)
            with autocast(enabled=self._use_amp):
                logits = self.strategy.forward(batch)
                loss   = self.criterion(logits, labels) / accum

            self.scaler.scale(loss).backward()

            if (step + 1) % accum == 0 or (step + 1) == len(loader):
                self.scaler.unscale_(self.optimizer)
                nn.utils.clip_grad_norm_(self.model.parameters(), max_norm)
                self.scaler.step(self.optimizer)
                self.scaler.update()
                if self._scheduler is not None:
                    self._scheduler.step()
                self.optimizer.zero_grad()

            total += loss.item() * accum
            steps += 1

            if step % 100 == 0:
                lr = self.optimizer.param_groups[0]["lr"]
                print(
                    f"  step {step + 1:>4}/{len(loader)} "
                    f"loss={loss.item() * accum:.4f}  lr={lr:.2e}"
                )

        return total / max(steps, 1)

    def _val_epoch(
        self,
        loader: DataLoader,
        label2answer: Dict[int, str],
    ) -> Tuple[float, Dict[str, float]]:
        self.model.eval()
        total, steps = 0.0, 0
        all_preds, all_labels = [], []

        with torch.no_grad():
            for batch in loader:
                labels = batch["answer_label"].to(self.device)
                with autocast(enabled=self._use_amp):
                    logits = self.strategy.forward(batch)
                    loss   = self.criterion(logits, labels)
                all_preds.extend(logits.argmax(-1).cpu().tolist())
                all_labels.extend(labels.cpu().tolist())
                total += loss.item()
                steps += 1

        metrics = Evaluator().compute(all_preds, all_labels, label2answer)
        return total / max(steps, 1), metrics

    # -----------------------------------------------------------------------
    # Inference
    # -----------------------------------------------------------------------

    def generate_predictions(
        self,
        loader: DataLoader,
        label2answer: Dict[int, str],
        output_path: str,
        experiment_id: str = "final",
    ) -> pd.DataFrame:
        self.model.eval()
        records = []

        with torch.no_grad():
            for batch in loader:
                preds = self.strategy.predict(batch)
                for i, p in enumerate(preds.cpu().tolist()):
                    records.append({
                        "id"           : str(uuid.uuid4()),
                        "experiment_id": experiment_id,
                        "image_url"    : batch["image_id"][i],
                        "question"     : batch["question"][i],
                        "answer"       : label2answer.get(p, "unknown"),
                        "ground_truth" : batch["answer"][i],
                        "timestamp"    : datetime.now(tz=timezone.utc).isoformat(),
                    })

        df = pd.DataFrame(records)
        os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
        df.to_csv(output_path, index=False)
        print(f"[Manager] {len(df)} predictions -> '{output_path}'")
        return df

    # -----------------------------------------------------------------------
    # Checkpointing
    # -----------------------------------------------------------------------

    def _save_ckpt(self, epoch: int, val_acc: float) -> None:
        """Save only when val accuracy improves; delete previous best first."""
        if val_acc <= self._best_val_acc:
            print(
                f"[Manager] Epoch {epoch}: acc={val_acc:.4f} <= best "
                f"{self._best_val_acc:.4f} -- checkpoint skipped."
            )
            return
        self._best_val_acc = val_acc
        path = os.path.join(
            self.config["checkpoint_dir"],
            f"{self.strategy.get_name()}_best.pt",
        )
        # Delete old best BEFORE writing new one to stay within disk limits
        if self._best_ckpt_path and os.path.exists(self._best_ckpt_path):
            os.remove(self._best_ckpt_path)
        torch.save({
            "epoch"               : epoch,
            "model_state_dict"    : self.model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "val_acc"             : val_acc,
        }, path)
        self._best_ckpt_path = path
        print(f"[Manager] Best checkpoint (epoch {epoch}, acc={val_acc:.4f}): '{path}'")

    def load_checkpoint(self, path: str) -> int:
        ckpt = torch.load(path, map_location=self.device)
        self.model.load_state_dict(ckpt["model_state_dict"])
        self.optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        epoch = ckpt.get("epoch", 0)
        print(f"[Manager] Resumed from checkpoint (epoch {epoch})")
        return epoch


print("VQAManager defined.")

VQAManager defined.


In [13]:
# ============================================================
# STEP 1: Dataset Exploration
# ============================================================

print("=" * 55, "\nDataset Exploration\n" + "=" * 55)

_train_raw = pd.read_csv(os.path.join(DATASET_DIR, "data_train.csv"))
_eval_raw  = pd.read_csv(os.path.join(DATASET_DIR, "data_eval.csv"))

with open(os.path.join(DATASET_DIR, "answer_space.txt")) as _f:
    _space = set(l.strip() for l in _f if l.strip())

print(f"Train samples : {len(_train_raw):,}")
print(f"Eval  samples : {len(_eval_raw):,}")
print(f"Answer space  : {len(_space)} classes")

_counts = Counter(_train_raw.answer.astype(str))
print(f"\nUnique train answers      : {len(_counts)}")

_not_in_space = [a for a in _counts if a not in _space]
print(f"Answers NOT in space      : {len(_not_in_space)} (all comma-separated)")
print(f"  All resolvable via split: {all(any(p.strip() in _space for p in a.split(',')) for a in _not_in_space)}")

_multi_samples = _train_raw[_train_raw.answer.astype(str).str.contains(",")]
print(f"\nMulti-label train samples : {len(_multi_samples):,}  "
      f"({100 * len(_multi_samples) / len(_train_raw):.1f}%)")

print(f"\nTop 10 answer frequencies:")
for ans, cnt in _counts.most_common(10):
    bar = "#" * int(cnt / _counts.most_common(1)[0][1] * 30)
    print(f"  {ans:<28} {cnt:>5}  {bar}")

print(f"\nClass imbalance: max={_counts.most_common(1)[0][1]} / "
      f"min={min(_counts.values())} samples")
print(f"Singletons (count=1)      : {sum(1 for c in _counts.values() if c == 1)}")

del _train_raw, _eval_raw, _not_in_space, _multi_samples, _counts

Dataset Exploration
Train samples : 9,974
Eval  samples : 2,494
Answer space  : 582 classes

Unique train answers      : 1260
Answers NOT in space      : 750 (all comma-separated)
  All resolvable via split: True

Multi-label train samples : 990  (9.9%)

Top 10 answer frequencies:
  2                              442  ##############################
  table                          346  #######################
  chair                          293  ###################
  3                              257  #################
  window                         227  ###############
  white                          213  ##############
  photo                          204  #############
  cabinet                        199  #############
  sofa                           189  ############
  1                              188  ############

Class imbalance: max=442 / min=1 samples
Singletons (count=1)      : 806


In [14]:
# ============================================================
# STEP 2: Processor, DatasetProcessor, class weights
# ============================================================

model_choice = CONFIG["model_choice"]
print(f"Loading processor for model_choice='{model_choice}' ...")

if model_choice == "blip":
    processor = BlipProcessor.from_pretrained(CONFIG["pretrained_id"])
else:
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

dp = DatasetProcessor(
    data_dir    = DATASET_DIR,
    processor   = processor,
    batch_size  = CONFIG["batch_size"],
    num_workers = 2,
    max_length  = CONFIG["max_length"],
)

train_loader = dp.get_train_loader()
eval_loader  = dp.get_eval_loader()
num_classes  = dp.get_num_classes()
label2answer = dp.label2answer

print(f"\nClasses      : {num_classes}")
print(f"Train batches: {len(train_loader)}")
print(f"Eval  batches: {len(eval_loader)}")

print("\nComputing inverse-frequency class weights ...")
class_weights = compute_class_weights(
    dp.train_df, dp.answer2label, num_classes,
    cap=CONFIG.get("class_weight_cap", 3.0),
)
print(f"Weight range : [{class_weights.min():.3f}, {class_weights.max():.3f}]")
print(f"Mean weight  : {class_weights.mean():.3f}")

Loading processor for model_choice='blip' ...


preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]


Classes      : 582
Train batches: 1247
Eval  batches: 312

Computing inverse-frequency class weights ...
Weight range : [0.035, 5.000]
Mean weight  : 3.070


In [15]:
# ============================================================
# STEP 3: Initialise model strategy
# ============================================================

device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
freeze_vision = CONFIG.get("freeze_vision_epochs", 0) > 0

if CONFIG["model_choice"] == "blip":
    strategy = BLIPStrategy(freeze_vision=freeze_vision)
else:
    strategy = CLIPStrategy()

strategy.load_model(num_classes=num_classes, device=device)

print(f"\nStrategy '{strategy.get_name()}' ready on {device}.")
if freeze_vision:
    print(
        f"Vision encoder frozen for the first {CONFIG['freeze_vision_epochs']} "
        f"epochs, then unfrozen at {CONFIG['learning_rate'] * 0.1:.1e} LR."
    )
else:
    print("Vision encoder trained end-to-end from epoch 1.")

[BLIPStrategy] Loading 'Salesforce/blip-vqa-base' ...


model.safetensors:   0%|          | 0.00/1.54G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/788 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForQuestionAnswering LOAD REPORT from: Salesforce/blip-vqa-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 
text_encoder.embeddings.position_ids      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[BLIPStrategy] 299,275,394 trainable / 385,365,890 total params

Strategy 'BLIP' ready on cuda.
Vision encoder frozen for the first 5 epochs, then unfrozen at 4.0e-06 LR.


In [16]:
# ============================================================
# STEP 4: Logger, Manager, Training
# ============================================================

logger = ExperimentLogger(csv_path=EXPERIMENTS_CSV)

manager = VQAManager(
    strategy      = strategy,
    logger        = logger,
    config        = CONFIG,
    class_weights = class_weights,
)

print(f"\nStarting training for {CONFIG['epochs']} epochs ...")
history = manager.train(
    train_loader = train_loader,
    val_loader   = eval_loader,
    label2answer = label2answer,
)

best_acc   = max(history["accuracy"])
best_epoch = history["accuracy"].index(best_acc) + 1
print(f"\nTraining complete.")
print(f"Best accuracy: {best_acc:.4f} at epoch {best_epoch}")

[Manager] Device: cuda
[Manager] Trainable params: 299,275,394

Starting training for 15 epochs ...
[Manager] LR scheduler: 2340 total steps, 234 warmup steps
[Logger] Experiment started: d3227313-376d-423f-ab33-4e60967b1d84

Epoch 1/15
  step    1/1247 loss=14.4171  lr=0.00e+00
  step  101/1247 loss=13.1280  lr=2.05e-06
  step  201/1247 loss=20.1712  lr=4.27e-06
  step  301/1247 loss=7.5952  lr=6.32e-06
  step  401/1247 loss=8.9395  lr=8.55e-06
  step  501/1247 loss=7.5961  lr=1.06e-05
  step  601/1247 loss=7.2413  lr=1.28e-05
  step  701/1247 loss=8.2894  lr=1.49e-05
  step  801/1247 loss=9.7458  lr=1.71e-05
  step  901/1247 loss=7.9757  lr=1.91e-05
  step 1001/1247 loss=7.6647  lr=2.14e-05
  step 1101/1247 loss=8.2398  lr=2.34e-05
  step 1201/1247 loss=8.5391  lr=2.56e-05
[Manager] Best checkpoint (epoch 1, acc=0.0794): '/kaggle/working/outputs/checkpoints/BLIP_best.pt'
[Logger] Epoch  1 | train=10.4053  val=10.1946  acc=0.0794  f1=0.0616  bleu=0.0159

Epoch 2/15
  step    1/1247 lo

In [17]:
# ============================================================
# STEP 5: Training curves
# ============================================================

plot_training_curves(
    train_loss = history["train_loss"],
    val_loss   = history["val_loss"],
    accuracy   = history["accuracy"],
    output_dir = OUTPUT_DIR,
    model_name = strategy.get_name(),
)

# ============================================================
# STEP 6: Generate predictions (GeneratedAnswers table)
# ============================================================

predictions_path = os.path.join(OUTPUT_DIR, "predictions.csv")
pred_df = manager.generate_predictions(
    loader        = eval_loader,
    label2answer  = label2answer,
    output_path   = predictions_path,
    experiment_id = "final",
)

exact_match = (pred_df["answer"] == pred_df["ground_truth"].apply(
    lambda a: str(a).split(",")[0].strip()  # normalise multi-labels for comparison
)).mean()
print(f"\nFinal exact-match accuracy (normalised): {exact_match:.4f}")

print("\nSample predictions (image_url, question, ground_truth, answer):")
pred_df[["image_url", "question", "ground_truth", "answer"]].head(10)

Plot saved to '/kaggle/working/outputs/BLIP_training_curves.png'
[Manager] 2494 predictions -> '/kaggle/working/outputs/predictions.csv'

Final exact-match accuracy (normalised): 0.3849

Sample predictions (image_url, question, ground_truth, answer):


,image_url,question,ground_truth,answer
0,image399,what is the colour of the bag on the chair,pink,pink
1,image1341,what is at the right bottom,table,table
2,image1320,what are found on the rack,toy,decorative_plate
3,image529,what is left of printer,mirror,paper_tray
4,image201,what is the colour of television,black,black
5,image1439,what is on the dining table,ornamental_plant,vase
6,image477,what is the ball on the table,basketball,ball
7,image1362,how many drawers are there,4,6
8,image838,what is on the left side of the sink,towel,kitchen_utensils
9,image73,what is on the bed,"comforter, pillow",comforter


In [18]:
# ============================================================
# STEP 7: Final experiment summary
# ============================================================

print("Per-epoch experiment log:")
log_df = pd.read_csv(EXPERIMENTS_CSV)
print(log_df[["id", "model_name", "train_loss", "val_loss",
              "metrics", "timestamp"]].to_string(index=False))

# Find the best epoch
_parsed = log_df["metrics"].apply(json.loads)
best_idx = _parsed.apply(lambda m: m.get("accuracy", 0)).idxmax()
best_row = log_df.iloc[best_idx]
best_m   = json.loads(best_row["metrics"])

print(f"\n{'=' * 55}")
print(f"Best result  (epoch {best_m.get('epoch', '?')})")
print(f"  Accuracy   : {best_m.get('accuracy', 0):.4f}")
print(f"  F1 (wt avg): {best_m.get('f1', 0):.4f}")
print(f"  BLEU       : {best_m.get('bleu', 0):.4f}")
print(f"  Val loss   : {best_row['val_loss']:.4f}")

print(f"\nOutput files in '{OUTPUT_DIR}':")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.isfile(fpath):
        print(f"  {fname}  ({os.path.getsize(fpath):,} bytes)")

Per-epoch experiment log:
                                       id model_name  train_loss  val_loss                                                         metrics                        timestamp
 d3227313-376d-423f-ab33-4e60967b1d84_ep1       BLIP   10.405274 10.194552  {"accuracy": 0.0794, "f1": 0.0616, "bleu": 0.0159, "epoch": 1} 2026-03-10T02:02:32.943324+00:00
 d3227313-376d-423f-ab33-4e60967b1d84_ep2       BLIP    9.738129  9.732599  {"accuracy": 0.2073, "f1": 0.1743, "bleu": 0.0436, "epoch": 2} 2026-03-10T02:08:29.986258+00:00
 d3227313-376d-423f-ab33-4e60967b1d84_ep3       BLIP    9.337382  9.434119   {"accuracy": 0.2935, "f1": 0.273, "bleu": 0.0589, "epoch": 3} 2026-03-10T02:14:28.470549+00:00
 d3227313-376d-423f-ab33-4e60967b1d84_ep4       BLIP    8.875979  9.216045  {"accuracy": 0.3232, "f1": 0.3072, "bleu": 0.0658, "epoch": 4} 2026-03-10T02:20:23.551539+00:00
 d3227313-376d-423f-ab33-4e60967b1d84_ep5       BLIP    8.374250  9.097191  {"accuracy": 0.3304, "f1": 0.3203, "bl